[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/22-reasoning-models.ipynb)

# Module 22: Reasoning Models

Companion notebook to [Module 22](https://transformers-and-math.drmhse.com/modules/22-reasoning-models) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
rng = np.random.default_rng(0)

## Part 1 — Spending samples at answer time (section 22.2)
A simulated solver whose single answers are right with probability $p$. Compare best-of-$n$ with a perfect verifier and majority voting, when wrong answers are scattered over four values.

In [ ]:
def simulate(p, n, trials=20_000, wrong_values=4):
    right = rng.random((trials, n)) < p
    answers = np.where(right, 0, 1 + rng.integers(0, wrong_values, (trials, n)))
    best_of_n = right.any(1).mean()
    counts = np.stack([(answers == v).sum(1) for v in range(wrong_values + 1)], 1)
    vote = (counts[:, 0] > counts[:, 1:].max(1)).mean()      # ties count as wrong
    return best_of_n, vote

for p in (0.3, 0.6):
    for n in (1, 4, 16):
        b, v = simulate(p, n)
        print(f'p={p}  n={n:2}:  best-of-n (perfect verifier) {b:.2f}   majority vote {v:.2f}   formula 1−(1−p)^n {1 - (1 - p) ** n:.2f}')

## Part 2 — RL with a verifiable reward, trained with GRPO (section 22.3)
A tiny policy must answer "what is $a + b$ (mod 10)?" for single digits $a, b$. It is never shown the answer, only rewarded 1 when its sampled answer is right. Advantages are relative to a group of 8 samples per question.

In [ ]:
torch.manual_seed(0)
policy = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 10))
opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
G = 8

def encode(a, b):
    return torch.cat([F.one_hot(a, 10), F.one_hot(b, 10)], -1).float()

def accuracy():
    a, b = torch.arange(10).repeat_interleave(10), torch.arange(10).repeat(10)
    with torch.no_grad():
        return (policy(encode(a, b)).argmax(-1) == (a + b) % 10).float().mean().item()

for step in range(600):
    a, b = torch.randint(0, 10, (32,)), torch.randint(0, 10, (32,))
    logits = policy(encode(a, b))                                   # (32, 10)
    dist = torch.distributions.Categorical(logits=logits)
    samples = dist.sample((G,))                                     # (G, 32): a group per question
    reward = (samples == (a + b) % 10).float()
    adv = (reward - reward.mean(0)) / (reward.std(0, unbiased=False) + 1e-6)
    loss = -(adv * dist.log_prob(samples)).mean()                   # policy gradient with group-relative advantages
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 150 == 0:
        print(f'step {step:3}  mean reward {reward.mean():.2f}  greedy accuracy on all 100 questions {accuracy():.0%}')
print(f'final greedy accuracy {accuracy():.0%}')

A group where every sample is right (or every one wrong) has all advantages zero, and teaches nothing:

In [ ]:
r = torch.tensor([1., 1., 1., 1.])
print('advantages of an all-correct group:', ((r - r.mean()) / (r.std(unbiased=False) + 1e-6)).tolist())
r = torch.tensor([1., 0., 0., 1., 0., 0., 0., 0.])
print('advantages of a 2-of-8 group:', [round(v, 2) for v in ((r - r.mean()) / r.std(unbiased=False)).tolist()])

## Your turn
1. In Part 1, make every wrong answer the same value (`wrong_values=1`) with $p = 0.3$. What does majority voting converge to?
2. In Part 2, replace the group-relative advantage with the raw reward. Does training still work? How does its speed compare?